# Hướng Dẫn Chạy Notebook Dành Riêng Cho KAGGLE

1. Upload thư mục Data lên Kaggle Dataset.\n2. Bấm Add Data bên góc phải để chèn vào Notebook.\n3. Chạy Run All. Model đã cài sẵn MF1 Score và Early Stopping.

In [ ]:
# Cài đặt thư viện wandb và scikit-learn
!pip install wandb scikit-learn -q

import os
# Tự động chèn và cấu hình API Key cho Weights & Biases
os.environ['WANDB_API_KEY'] = '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'
os.environ['WANDB_SILENT'] = 'true'
from tqdm import tqdm


In [ ]:
import wandb
import random

run = wandb.init(
    entity="giabao240806-fpt-university",
    project="Đồng cam mất ngủ",
    config={
        "learning_rate_pretrain": 1e-4,
        "learning_rate_finetune": 1e-5,
        "architecture": "SleepTransformer",
        "dataset": "Sleep-EDF",
        "epochs_pretrain": 100,
        "epochs_finetune": 100,
        "batch_size" : 32,
        "early_stopping_patience": 20
    },
    name=f"SleepTransformer",
    save_code=True
)


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score, cohen_kappa_score
import warnings
warnings.filterwarnings('ignore')
import os
import glob

# ====================================================================
# CẤU HÌNH ĐƯỜNG DẪN KAGGLE
# ====================================================================

DATA_DIR = '/kaggle/input/datasets/lcgiabo/ng-cam-mt-ng/SC_Data'
SAVE_DIR = '/kaggle/working/SLEEP_MODELS/'
os.makedirs(SAVE_DIR, exist_ok=True)

# Hyperparameters
BATCH_SIZE_PRETRAIN = 128
BATCH_SIZE_FINETUNE = 32
SEQ_LEN = 20
EPOCHS_PRETRAIN = 100
EPOCHS_FINETUNE = 100
LR_PRETRAIN = 1e-4
LR_FINETUNE = 1e-5
EARLY_STOPPING_PATIENCE = 20
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", DEVICE)
from tqdm import tqdm


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float) * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        # x: (Batch, Seq_Len, d_model)
        x = x + self.pe[:, :x.size(1)]
        return x

class CNNFrontEnd(nn.Module):
    def __init__(self, in_channels=1, out_channels=128, T=29):
        super(CNNFrontEnd, self).__init__()
        self.features = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=50, stride=6, bias=False),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=8, stride=8),
            nn.Dropout(0.5),
            
            nn.Conv1d(64, 128, kernel_size=8, stride=1, bias=False),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Conv1d(128, out_channels, kernel_size=8, stride=1, bias=False),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(),
            nn.Dropout(0.5)
        )
        self.pool = nn.AdaptiveAvgPool1d(T)
        
    def forward(self, x):
        # x: (B, 1, 3000)
        feat = self.features(x)  # (B, out_channels, L_out)
        feat = self.pool(feat)   # (B, out_channels, T)
        feat = feat.transpose(1, 2)  # (B, T, out_channels)
        return feat

class AttentionPooler(nn.Module):
    def __init__(self, d_model, attention_size=64):
        super(AttentionPooler, self).__init__()
        self.Wa = nn.Linear(d_model, attention_size)
        self.ae = nn.Parameter(torch.randn(attention_size, 1))
        
    def forward(self, x):
        # x: (Batch, Seq_Len, d_model)
        at = torch.tanh(self.Wa(x))  # (Batch, Seq_Len, attention_size)
        scores = torch.matmul(at, self.ae).squeeze(-1)  # (Batch, Seq_Len)
        alpha = torch.softmax(scores, dim=-1).unsqueeze(-1)  # (Batch, Seq_Len, 1)
        x_pooled = torch.sum(alpha * x, dim=1)  # (Batch, d_model)
        return x_pooled

class SleepTransformer(nn.Module):
    def __init__(self, in_channels=1, num_classes=5, d_model=128, nhead=8, 
                 d_ff=1024, num_epoch_layers=4, num_sequence_layers=4, 
                 dropout=0.1, attention_size=64):
        super(SleepTransformer, self).__init__()
        
        self.d_model = d_model
        self.cnn_frontend = CNNFrontEnd(in_channels=in_channels, out_channels=d_model, T=29)
        self.pos_encoder_epoch = PositionalEncoding(d_model=d_model)
        self.pos_encoder_seq = PositionalEncoding(d_model=d_model)
        
        epoch_encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_ff, 
            dropout=dropout, activation='relu', batch_first=True
        )
        self.epoch_transformer = nn.TransformerEncoder(epoch_encoder_layer, num_layers=num_epoch_layers)
        
        self.attention_pool = AttentionPooler(d_model=d_model, attention_size=attention_size)
        
        self.pretrain_classifier = nn.Linear(d_model, num_classes)
        
        seq_encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_ff, 
            dropout=dropout, activation='relu', batch_first=True
        )
        self.seq_transformer = nn.TransformerEncoder(seq_encoder_layer, num_layers=num_sequence_layers)
        
        self.finetune_classifier = nn.Sequential(
            nn.Linear(d_model, 1024),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(1024, 1024),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(1024, num_classes)
        )

    def freeze_cnn(self):
        for param in self.cnn_frontend.parameters():
            param.requires_grad = False
            
    def unfreeze_cnn(self):
        for param in self.cnn_frontend.parameters():
            param.requires_grad = True

    def _epoch_feature_extraction(self, x):
        cnn_features = self.cnn_frontend(x) 
        cnn_features = self.pos_encoder_epoch(cnn_features)
        epoch_features = self.epoch_transformer(cnn_features) 
        pooled_features = self.attention_pool(epoch_features) 
        return pooled_features

    def forward(self, x, mode="finetune"):
        if mode == "pretrain":
            if x.dim() == 4:
                x = x.squeeze(1)  
            
            pooled_features = self._epoch_feature_extraction(x) 
            return self.pretrain_classifier(pooled_features) 
            
        elif mode == "finetune":
            B, Seq_Len, C, L = x.shape
            x = x.view(B * Seq_Len, C, L)  
            
            pooled_features = self._epoch_feature_extraction(x)  
            seq_features = pooled_features.view(B, Seq_Len, self.d_model)  
            seq_features = self.pos_encoder_seq(seq_features)
            
            seq_out = self.seq_transformer(seq_features)  
            return self.finetune_classifier(seq_out)  
            
        else:
            raise ValueError("mode must be 'pretrain' or 'finetune'")

In [ ]:
class SingleEpochDataset(Dataset):
    def __init__(self, file_paths, oversample=True):
        self.x = []
        self.y = []
        for p in file_paths:
            data = np.load(p)
            self.x.append(data['x'])
            self.y.append(data['y'])
        
        if not self.x:
            self.x = torch.empty((0, 1, 3000), dtype=torch.float32)
            self.y = torch.empty((0,), dtype=torch.long)
            self.indices = []
            return
        
        self.x = np.concatenate(self.x, axis=0)
        self.y = np.concatenate(self.y, axis=0)
        self.indices = np.arange(len(self.y))
        
        if oversample and len(self.y) > 0:
            classes, counts = np.unique(self.y, return_counts=True)
            max_count = np.max(counts)
            oversampled_indices = []
            for cls in classes:
                idx = np.where(self.y == cls)[0]
                idx_resampled = np.random.choice(idx, max_count, replace=True)
                oversampled_indices.append(idx_resampled)
            self.indices = np.concatenate(oversampled_indices, axis=0)
            
        self.x = torch.tensor(self.x, dtype=torch.float32)
        self.y = torch.tensor(self.y, dtype=torch.long)
        
    def __len__(self):
        return len(self.indices)
        
    def __getitem__(self, i):
        idx = self.indices[i]
        return self.x[idx], self.y[idx]
class SequenceDataset(Dataset):
    def __init__(self, file_paths, seq_len=20, stride=20):
        self.seq_len = seq_len
        self.data_cache = []
        self.windows_map = []
        
        for idx, p in enumerate(file_paths):
            data = np.load(p)
            self.data_cache.append({
                'x': torch.tensor(data['x'], dtype=torch.float32),
                'y': torch.tensor(data['y'], dtype=torch.long)
            })
            
            num_epochs = len(data['y'])
            for i in range(0, num_epochs - seq_len + 1, stride):
                self.windows_map.append((idx, i))

    def __len__(self):
        return len(self.windows_map)
        
    def __getitem__(self, idx):
        file_idx, start_idx = self.windows_map[idx]
        data = self.data_cache[file_idx]
        x_seq = data['x'][start_idx : start_idx + self.seq_len]
        y_seq = data['y'][start_idx : start_idx + self.seq_len]
        return x_seq, y_seq


In [ ]:
def train_step(model, dataloader, optimizer, criterion, mode):
    model.train()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    
    for inputs, labels in tqdm(dataloader, desc=f"  - {mode}", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(inputs, mode=mode)
        
        if mode == "finetune":
            outputs = outputs.view(-1, outputs.size(-1))
            labels = labels.view(-1)
            
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
    acc = 100. * correct / total
    mf1 = 100. * f1_score(all_labels, all_preds, average='macro')
    kappa = cohen_kappa_score(all_labels, all_preds)
    return total_loss / len(dataloader), acc, mf1, kappa

def val_step(model, dataloader, criterion, mode):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc=f"  - {mode}", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs, mode=mode)
            
            if mode == "finetune":
                outputs = outputs.view(-1, outputs.size(-1))
                labels = labels.view(-1)
                
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    acc = 100. * correct / total
    mf1 = 100. * f1_score(all_labels, all_preds, average='macro')
    kappa = cohen_kappa_score(all_labels, all_preds)
    return total_loss / len(dataloader), acc, mf1, kappa


In [ ]:
def main():
    all_files = glob.glob(os.path.join(DATA_DIR, "*.npz"))
    if len(all_files) == 0:
        print(f"Chưa có data trong {DATA_DIR}. Vui lòng kiểm tra lại cấu trúc thư mục Kaggle Dataset!")
        return

    print("Khởi tạo K-Fold...")
    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    train_idx, val_idx = next(kf.split(all_files))
    
    train_files = [all_files[i] for i in train_idx]
    val_files = [all_files[i] for i in val_idx]
    
    model = SleepTransformer(in_channels=1, num_classes=5).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    
    # -------------------------------------
    # BƯỚC 1: PRE-TRAINING
    # -------------------------------------
    print("\n=== BẮT ĐẦU PRE-TRAINING (1-STEP) ===")
    
    train_ds_1 = SingleEpochDataset(train_files, oversample=True)
    val_ds_1 = SingleEpochDataset(val_files, oversample=False)
    
    train_loader_1 = DataLoader(train_ds_1, batch_size=BATCH_SIZE_PRETRAIN, shuffle=True, num_workers=0, pin_memory=True)
    val_loader_1 = DataLoader(val_ds_1, batch_size=BATCH_SIZE_PRETRAIN, shuffle=False, num_workers=0, pin_memory=True)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=LR_PRETRAIN, weight_decay=1e-4)
    
    best_val_mf1_1 = 0
    patience_counter_1 = 0
    
    for epoch in range(EPOCHS_PRETRAIN):
        train_loss, train_acc, train_mf1, train_kappa = train_step(model, train_loader_1, optimizer, criterion, mode="pretrain")
        val_loss, val_acc, val_mf1, val_kappa = val_step(model, val_loader_1, criterion, mode="pretrain")
        
        wandb.log({"epoch": epoch+1, 
                   "pretrain_train_loss": train_loss, "pretrain_val_f1": val_mf1, 
                   "pretrain_val_kappa": val_kappa, "pretrain_val_acc": val_acc})
                   
        print(f"Epoch [{epoch+1}/{EPOCHS_PRETRAIN}] Pretrain - Loss: {train_loss:.4f} - Val Acc: {val_acc:.2f}% - Val MF1: {val_mf1:.2f}%")
        
        # Early Stopping & Checkpoint (Dựa trên MF1 thay vì Accuracy)
        if val_mf1 > best_val_mf1_1:
            best_val_mf1_1 = val_mf1
            patience_counter_1 = 0
            torch.save(model.state_dict(), os.path.join(SAVE_DIR, "sleeptransformer_pretrain_best.pth"))
        else:
            patience_counter_1 += 1
            if patience_counter_1 >= EARLY_STOPPING_PATIENCE:
                print(f"Early Stopping kích hoạt ở Epoch {epoch+1} (Pretrain) do MF1 không tiến bộ!")
                break
                
    # -------------------------------------
    # BƯỚC 2: FINE-TUNING
    # -------------------------------------
    print("\n=== BẮT ĐẦU FINE-TUNING (2-STEP) ===")
    
    # Nạp lại trọng số xịn nhất của Step 1 trước khi sang Step 2
    model.load_state_dict(torch.load(os.path.join(SAVE_DIR, "sleeptransformer_pretrain_best.pth")))
    model.freeze_cnn()
    
    train_ds_2 = SequenceDataset(train_files, seq_len=SEQ_LEN, stride=1) # Đã đổi stride=1 nhờ LazyDataset
    val_ds_2 = SequenceDataset(val_files, seq_len=SEQ_LEN, stride=SEQ_LEN)
    
    train_loader_2 = DataLoader(train_ds_2, batch_size=BATCH_SIZE_FINETUNE, shuffle=True, num_workers=0, pin_memory=True)
    val_loader_2 = DataLoader(val_ds_2, batch_size=BATCH_SIZE_FINETUNE, shuffle=False, num_workers=0, pin_memory=True)
    
    optimizer2 = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_FINETUNE)
    
    best_val_mf1_2 = 0
    best_val_acc_2 = 0
    best_val_kappa_2 = 0
    patience_counter_2 = 0
    
    for epoch in range(EPOCHS_FINETUNE):
        train_loss, train_acc, train_mf1, train_kappa = train_step(model, train_loader_2, optimizer2, criterion, mode="finetune")
        val_loss, val_acc, val_mf1, val_kappa = val_step(model, val_loader_2, criterion, mode="finetune")
        
        # Ghi log đúng luật WandB để chuẩn hóa
        wandb.log({"epoch": epoch+1, 
                   "train_loss": train_loss, "val_f1": val_mf1,
                   "val_kappa": val_kappa, "val_acc": val_acc})
                   
        print(f"Epoch [{epoch+1}/{EPOCHS_FINETUNE}] Finetune - Loss: {train_loss:.4f} - Val Acc: {val_acc:.2f}% - Val MF1: {val_mf1:.2f}%")
        
        # Early Stopping & Checkpoint
        if val_mf1 > best_val_mf1_2:
            best_val_mf1_2 = val_mf1
            best_val_acc_2 = val_acc
            best_val_kappa_2 = val_kappa
            patience_counter_2 = 0
            torch.save(model.state_dict(), os.path.join(SAVE_DIR, "sleeptransformer_finetune_best.pth"))
            print("Đã lưu model tốt nhất dựa trên MF1 Score!")
        else:
            patience_counter_2 += 1
            if patience_counter_2 >= EARLY_STOPPING_PATIENCE:
                print(f"Early Stopping kích hoạt ở Epoch {epoch+1} (Finetune) do MF1 không tiến bộ!")
                break
                
    # Ghi log kết quả Final Fold cho nhóm tổng hợp
    wandb.log({"final_fold_acc": best_val_acc_2, 
               "final_fold_f1": best_val_mf1_2, 
               "final_fold_kappa": best_val_kappa_2})
            
    wandb.finish()
    print("✅ HOÀN TẤT TRAINING!")

main()
